In [1]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime
import random

print("Note: Using the built-in 'random' module to generate fake data.")

# --- 1. Define constants based on your sales_data.csv ---
UNIQUE_STORES = ['STR001', 'STR002', 'STR003', 'STR004', 'STR005', 'STR006', 'STR007', 'STR008', 'STR009', 'STR010']
UNIQUE_SKUS = ['SKU1001', 'SKU1002', 'SKU1003', 'SKU1004', 'SKU1005']
START_DATE = "2025-07-01"
END_DATE = "2026-03-31"

print(f"Generating data from {START_DATE} to {END_DATE}...")
print(f"For {len(UNIQUE_STORES)} stores and {len(UNIQUE_SKUS)} SKUs.")

# --- 2. Create base DataFrame with all combinations ---
# Create the date range
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='D')

# Create all combinations of date, store, and sku
index = pd.MultiIndex.from_product(
    [date_range, UNIQUE_STORES, UNIQUE_SKUS],
    names=["date", "store_id", "product_id"]
)
# Convert MultiIndex to DataFrame
forecast_df = pd.DataFrame(index=index).reset_index()
# Convert date column to datetime64[ns] for merging
forecast_df['date'] = pd.to_datetime(forecast_df['date'])

print(f"Created base DataFrame with {len(forecast_df)} combinations.")

# --- 3. Load and prepare actual sales data ---
try:
    # Load the provided sales data
    sales_df = pd.read_csv("sales_data.csv")
    # Convert sales data date column to datetime64[ns] for merging
    sales_df['date'] = pd.to_datetime(sales_df['date'])
    # Rename 'sku' to 'product_id' to match your forecast table model
    sales_df.rename(columns={'sku': 'product_id'}, inplace=True)
    
    # Aggregate data: sum units_sold for any duplicate date/store/product entries
    actuals_df = sales_df.groupby(['date', 'store_id', 'product_id'])['units_sold'].sum().reset_index()
    
    print("Loaded and processed actuals from sales_data.csv.")

    # --- 4. Merge actuals into the base forecast DataFrame ---
    # Use a left merge to keep all dates in our desired range
    final_df = pd.merge(
        forecast_df,
        actuals_df,
        on=['date', 'store_id', 'product_id'],
        how='left'
    )
    
    # Rename 'units_sold' to 'actual' to match the model
    # 'actual' column will have NaN where no sales data was found
    final_df.rename(columns={'units_sold': 'actual'}, inplace=True)
    
    print("Merged base combinations with actuals.")

    # --- 5. Generate remaining fake data ---
    num_rows = len(final_df)
    
    # Generate a single forecast_log_id for this batch
    log_id = uuid.uuid4()
    final_df['forecast_log_id'] = log_id
    
    # Function to generate a 'predicted' value
    def generate_prediction(actual_value):
        if pd.isna(actual_value):
            # No actual data, predict a random amount
            return round(random.uniform(5.0, 300.0), 2)
        else:
            # Actual data exists, predict a value +/- 20% of actual
            # Ensure prediction is non-negative
            prediction = actual_value * random.uniform(0.8, 1.2)
            return round(max(0.0, prediction), 2)

    final_df['predicted'] = final_df['actual'].apply(generate_prediction)
    
    # Add 'id' and 'created_at'
    final_df['id'] = range(1, num_rows + 1)
    final_df['created_at'] = datetime.utcnow()
    
    # Convert date back to string format as in your model's to_dict
    final_df['date'] = final_df['date'].dt.strftime("%Y-%m-%d")
    
    # --- 6. Format and output ---
    # Reorder columns to match your model
    column_order = [
        "id",
        "forecast_log_id",
        "date",
        "store_id",
        "product_id",
        "predicted",
        "actual",
        "created_at"
    ]
    final_df = final_df[column_order]

    # Save to CSV
    output_filename = "fake_forecast_data.csv"
    final_df.to_csv(output_filename, index=False)
    
    print(f"\nSuccessfully generated {num_rows} rows of fake forecast data.")
    print(f"Data saved to {output_filename}")

    print("\n--- Final DataFrame Info ---")
    final_df.info()

    print("\n--- First 5 Rows of Generated Data ---")
    print(final_df.head())

    print("\n--- Sample Rows Where Actuals Were Found ---")
    print(final_df[final_df['actual'].notna()].head())

except FileNotFoundError:
    print("Error: sales_data.csv not found.")
except Exception as e:
    print(f"An error occurred during data generation: {e}")

Note: Using the built-in 'random' module to generate fake data.
Generating data from 2025-07-01 to 2026-03-31...
For 10 stores and 5 SKUs.
Created base DataFrame with 13700 combinations.
Loaded and processed actuals from sales_data.csv.
Merged base combinations with actuals.

Successfully generated 13700 rows of fake forecast data.
Data saved to fake_forecast_data.csv

--- Final DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13700 entries, 0 to 13699
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id               13700 non-null  int64         
 1   forecast_log_id  13700 non-null  object        
 2   date             13700 non-null  object        
 3   store_id         13700 non-null  object        
 4   product_id       13700 non-null  object        
 5   predicted        13700 non-null  float64       
 6   actual           4950 non-null   float64       
 7   created_a

C:\Users\sudee\AppData\Local\Temp\ipykernel_16524\258594425.py:85: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  final_df['created_at'] = datetime.utcnow()


In [2]:
import pandas as pd
import numpy as np

try:
    # Load the file created in the previous step, which contains the errors
    df_metrics = pd.read_csv("fake_forecast_data.csv")
    print("Loaded 'fake_forecast_data.csv' to calculate metrics.")
    
    # The file should already be filtered (no NaN 'actuals'), but we check just in case.
    if df_metrics['actual'].isnull().any():
        print("Warning: Found null actuals. Filtering them out...")
        df_metrics.dropna(subset=['actual'], inplace=True)

    if df_metrics.empty:
        print("No data available to calculate metrics.")
    else:
        # --- 1. Calculate MAE (Mean Absolute Error) ---
        # The 'absolute_error' column was already calculated
        mae = df_metrics['absolute_error'].mean()

        # --- 2. Calculate WMAPE (Weighted Mean Absolute Percentage Error) ---
        sum_absolute_error = df_metrics['absolute_error'].sum()
        sum_actual = df_metrics['actual'].sum()

        wmape_percent = np.nan
        if sum_actual > 0:
            wmape_raw = sum_absolute_error / sum_actual
            wmape_percent = wmape_raw * 100
        else:
            print("Sum of actuals is 0, cannot calculate WMAPE.")

        print("\n--- Forecast Accuracy Metrics ---")
        print(f"Mean Absolute Error (MAE): {mae:.4f}")
        print(f"Weighted Mean Absolute Percentage Error (WMAPE): {wmape_percent:.2f}%")
        print(f"(Based on {len(df_metrics)} rows)")

except FileNotFoundError:
    print("Error: 'fake_forecast_data.csv' not found.")
    print("Please ensure the previous step ran successfully and created this file.")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded 'fake_forecast_data.csv' to calculate metrics.
An error occurred: 'absolute_error'


In [3]:
import pandas as pd
import numpy as np

try:
    # Load the file, which has the columns you listed
    df = pd.read_csv("fake_forecast_data.csv")
    print("Loaded 'fake_forecast_data.csv' to calculate metrics.")
    
    # 1. Filter out rows where 'actual' is null, as metrics can't be calculated
    df_metrics = df.dropna(subset=['actual']).copy()
    print(f"Filtered data to {len(df_metrics)} rows where 'actual' is not null.")


    if df_metrics.empty:
        print("No data with actuals available to calculate metrics.")
    else:
        # 2. Calculate the 'absolute_error'
        # This is the absolute difference between 'actual' and 'predicted'
        df_metrics['absolute_error'] = np.abs(df_metrics['actual'] - df_metrics['predicted'])
        
        # --- 3. Calculate MAE (Mean Absolute Error) ---
        # MAE = Mean of the absolute errors
        mae = df_metrics['absolute_error'].mean()

        # --- 4. Calculate WMAPE (Weighted Mean Absolute Percentage Error) ---
        # WMAPE = Sum of Absolute Errors / Sum of Actuals
        sum_absolute_error = df_metrics['absolute_error'].sum()
        sum_actual = df_metrics['actual'].sum()

        wmape_percent = np.nan
        if sum_actual > 0:
            wmape_raw = sum_absolute_error / sum_actual
            wmape_percent = wmape_raw * 100
        else:
            print("Sum of actuals is 0, cannot calculate WMAPE.")

        print("\n--- Forecast Accuracy Metrics ---")
        print(f"Mean Absolute Error (MAE): {mae:.4f}")
        print(f"Weighted Mean Absolute Percentage Error (WMAPE): {wmape_percent:.2f}%")

except FileNotFoundError:
    print("Error: 'fake_forecast_data.csv' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded 'fake_forecast_data.csv' to calculate metrics.
Filtered data to 4950 rows where 'actual' is not null.

--- Forecast Accuracy Metrics ---
Mean Absolute Error (MAE): 9.2115
Weighted Mean Absolute Percentage Error (WMAPE): 9.98%


In [5]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime
import random

print("--- Starting Full Data Generation (v3) ---")

# --- Constants ---
UNIQUE_STORES = ['STR001', 'STR002', 'STR003', 'STR004', 'STR005', 'STR006', 'STR007', 'STR008', 'STR009', 'STR010']
UNIQUE_SKUS = ['SKU1001', 'SKU1002', 'SKU1003', 'SKU1004', 'SKU1005']
FORECAST_START_DATE = "2025-07-01"
FORECAST_END_DATE = "2026-03-31"

# --- PART 1: Create a new, expanded set of 'actuals' ---
print("Part 1: Creating new expanded 'actuals' data...")
try:
    # Load the original sales data
    sales_df = pd.read_csv("sales_data.csv")
    sales_df['date'] = pd.to_datetime(sales_df['date'])
    sales_df.rename(columns={'sku': 'product_id'}, inplace=True)
    print(f"Loaded {len(sales_df)} rows from sales_data.csv.")

    # --- Generate new fake sales data from Oct 8 to Nov 10 ---
    NEW_SALES_START_DATE = "2025-10-08"
    NEW_SALES_END_DATE = "2025-11-10"
    print(f"Generating new fake actuals from {NEW_SALES_START_DATE} to {NEW_SALES_END_DATE}...")
    
    new_date_range = pd.date_range(start=NEW_SALES_START_DATE, end=NEW_SALES_END_DATE, freq='D')
    
    new_sales_index = pd.MultiIndex.from_product(
        [new_date_range, UNIQUE_STORES, UNIQUE_SKUS],
        names=["date", "store_id", "product_id"]
    )
    new_sales_df = pd.DataFrame(index=new_sales_index).reset_index()
    
    # Generate random 'units_sold' for this new period
    new_sales_df['units_sold'] = [random.randint(5, 300) for _ in range(len(new_sales_df))]
    print(f"Generated {len(new_sales_df)} new fake sales rows.")
    
    # Combine original sales with new fake sales
    combined_sales_df = pd.concat([sales_df, new_sales_df], ignore_index=True)
    
    # Aggregate data
    combined_actuals_df = combined_sales_df.groupby(['date', 'store_id', 'product_id'])['units_sold'].sum().reset_index()
    print(f"Total 'actuals' data rows (original + new): {len(combined_actuals_df)}")

    # --- PART 2: Generate the new forecast file (v3) ---
    print("\nPart 2: Generating new forecast file 'fake_forecast_data_v3.csv'...")
    
    # Create the base forecast DataFrame (July 2025 - March 2026)
    forecast_date_range = pd.date_range(start=FORECAST_START_DATE, end=FORECAST_END_DATE, freq='D')
    forecast_index = pd.MultiIndex.from_product(
        [forecast_date_range, UNIQUE_STORES, UNIQUE_SKUS],
        names=["date", "store_id", "product_id"]
    )
    final_df = pd.DataFrame(index=forecast_index).reset_index()
    final_df['date'] = pd.to_datetime(final_df['date'])
    
    # Merge the new, expanded 'actuals' data
    final_df = pd.merge(
        final_df,
        combined_actuals_df,
        on=['date', 'store_id', 'product_id'],
        how='left'
    )
    final_df.rename(columns={'units_sold': 'actual'}, inplace=True)
    print("Merged new expanded actuals into forecast timeline.")
    
    # --- Define the 'v2' prediction logic ---
    def generate_prediction_v2(row):
        actual_value = row['actual']
        date = row['date'] # This is a datetime object
        
        special_start_date = pd.to_datetime("2025-08-01")
        special_end_date = pd.to_datetime("2025-10-31")

        if pd.isna(actual_value):
            # No actual data, predict a random amount
            return round(random.uniform(5.0, 300.0), 2)
        else:
            # Actual data exists, apply logic based on date
            if special_start_date <= date <= special_end_date:
                # Aug-Oct logic: bias of -15%, std dev of 5%
                error_percentage = np.random.normal(loc=-0.15, scale=0.05)
                prediction = actual_value * (1 + error_percentage)
            else:
                # Logic for all other dates (including new Nov dates): uniform +/- 20%
                prediction = actual_value * random.uniform(0.8, 1.2)
                
            return round(max(0.0, prediction), 2)

    print("Applying 'v2' prediction logic to all rows...")
    final_df['predicted'] = final_df.apply(generate_prediction_v2, axis=1)

    # --- Add remaining columns and save ---
    num_rows = len(final_df)
    log_id = uuid.uuid4()
    final_df['forecast_log_id'] = log_id
    final_df['id'] = range(1, num_rows + 1)
    final_df['created_at'] = datetime.utcnow()
    
    final_df['date'] = final_df['date'].dt.strftime("%Y-%m-%d")
    
    column_order = [
        "id", "forecast_log_id", "date", "store_id", "product_id",
        "predicted", "actual", "created_at"
    ]
    final_df = final_df[column_order]

    output_filename = "fake_forecast_data_v3.csv"
    final_df.to_csv(output_filename, index=False)
    print(f"Successfully generated {num_rows} rows.")
    print(f"New data saved to {output_filename}")
    
except FileNotFoundError:
    print("Error: sales_data.csv not found.")
except Exception as e:
    print(f"An error occurred: {e}")

--- Starting Full Data Generation (v3) ---
Part 1: Creating new expanded 'actuals' data...
Loaded 20150 rows from sales_data.csv.
Generating new fake actuals from 2025-10-08 to 2025-11-10...
Generated 1700 new fake sales rows.
Total 'actuals' data rows (original + new): 21850

Part 2: Generating new forecast file 'fake_forecast_data_v3.csv'...
Merged new expanded actuals into forecast timeline.
Applying 'v2' prediction logic to all rows...
Successfully generated 13700 rows.
New data saved to fake_forecast_data_v3.csv


C:\Users\sudee\AppData\Local\Temp\ipykernel_16524\3643974232.py:101: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  final_df['created_at'] = datetime.utcnow()


In [6]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime
import random

print("--- Starting Full Data Generation (v5) with WHOLE NUMBERS ---")

# --- Constants ---
UNIQUE_STORES = ['STR001', 'STR002', 'STR003', 'STR004', 'STR005', 'STR006', 'STR007', 'STR008', 'STR009', 'STR010']
UNIQUE_SKUS = ['SKU1001', 'SKU1002', 'SKU1003', 'SKU1004', 'SKU1005']
FORECAST_START_DATE = "2025-07-01"
FORECAST_END_DATE = "2026-03-31"

# --- PART 1: Create a new, expanded set of 'actuals' (Same as v3/v4) ---
print("Part 1: Creating new expanded 'actuals' data...")
try:
    # Load the original sales data
    sales_df = pd.read_csv("sales_data.csv")
    sales_df['date'] = pd.to_datetime(sales_df['date'])
    sales_df.rename(columns={'sku': 'product_id'}, inplace=True)

    # --- Generate new fake sales data from Oct 8 to Nov 10 ---
    NEW_SALES_START_DATE = "2025-10-08"
    NEW_SALES_END_DATE = "2025-11-10"
    
    new_date_range = pd.date_range(start=NEW_SALES_START_DATE, end=NEW_SALES_END_DATE, freq='D')
    
    new_sales_index = pd.MultiIndex.from_product(
        [new_date_range, UNIQUE_STORES, UNIQUE_SKUS],
        names=["date", "store_id", "product_id"]
    )
    new_sales_df = pd.DataFrame(index=new_sales_index).reset_index()
    # Ensure new 'units_sold' are integers
    new_sales_df['units_sold'] = [random.randint(5, 300) for _ in range(len(new_sales_df))]
    
    # Combine original sales with new fake sales
    combined_sales_df = pd.concat([sales_df, new_sales_df], ignore_index=True)
    
    # Aggregate data
    combined_actuals_df = combined_sales_df.groupby(['date', 'store_id', 'product_id'])['units_sold'].sum().reset_index()
    print(f"Total 'actuals' data rows (original + new): {len(combined_actuals_df)}")

    # --- PART 2: Generate the new forecast file (v5) ---
    print("\nPart 2: Generating new forecast file 'fake_forecast_data_v5.csv'...")
    
    forecast_date_range = pd.date_range(start=FORECAST_START_DATE, end=FORECAST_END_DATE, freq='D')
    forecast_index = pd.MultiIndex.from_product(
        [forecast_date_range, UNIQUE_STORES, UNIQUE_SKUS],
        names=["date", "store_id", "product_id"]
    )
    final_df = pd.DataFrame(index=forecast_index).reset_index()
    final_df['date'] = pd.to_datetime(final_df['date'])
    
    # Merge the new, expanded 'actuals' data
    final_df = pd.merge(
        final_df,
        combined_actuals_df,
        on=['date', 'store_id', 'product_id'],
        how='left'
    )
    final_df.rename(columns={'units_sold': 'actual'}, inplace=True)
    print("Merged new expanded actuals into forecast timeline.")
    
    # --- Define the 'v5' REALISTIC WHOLE NUMBER prediction logic ---
    def generate_prediction_v5_wholenum(row):
        actual_value = row['actual']
        date = row['date'] # This is a datetime object
        
        special_start_date = pd.to_datetime("2025-08-01")
        special_end_date = pd.to_datetime("2025-10-31")

        if pd.isna(actual_value):
            # --- MODIFICATION 1 ---
            # No actual data, predict a random WHOLE NUMBER
            return random.randint(5, 300)
        else:
            # Actual data exists, apply logic based on date
            if special_start_date <= date <= special_end_date:
                # Aug-Oct: Mean error of 0%, std dev of 15%
                error_percentage = np.random.normal(loc=0.0, scale=0.15)
                prediction = actual_value * (1 + error_percentage)
            else:
                # Logic for all other dates: uniform +/- 20%
                prediction = actual_value * random.uniform(0.8, 1.2)
            
            # --- MODIFICATION 2 ---
            # Return a WHOLE NUMBER
            return int(round(max(0.0, prediction)))

    print("Applying 'v5' realistic whole number logic to all rows...")
    final_df['predicted'] = final_df.apply(generate_prediction_v5_wholenum, axis=1)

    # --- Add remaining columns and save ---
    num_rows = len(final_df)
    log_id = uuid.uuid4()
    final_df['forecast_log_id'] = log_id
    final_df['id'] = range(1, num_rows + 1)
    final_df['created_at'] = datetime.utcnow()
    
    final_df['date'] = final_df['date'].dt.strftime("%Y-%m-%d")
    
    column_order = [
        "id", "forecast_log_id", "date", "store_id", "product_id",
        "predicted", "actual", "created_at"
    ]
    final_df = final_df[column_order]

    output_filename = "fake_forecast_data_v5.csv"
    final_df.to_csv(output_filename, index=False)
    print(f"Successfully generated {num_rows} rows.")
    print(f"New data saved to {output_filename}")

    # --- PART 3: Calculate Metrics for the new file ---
    print("\nPart 3: Calculating metrics for 'fake_forecast_data_v5.csv'...")
    
    # Filter for rows where metrics can be calculated
    df_metrics = final_df.dropna(subset=['actual']).copy()
    print(f"Calculating metrics based on {len(df_metrics)} rows where 'actual' is not null.")

    if not df_metrics.empty:
        # Calculate 'absolute_error'
        df_metrics['absolute_error'] = np.abs(df_metrics['actual'] - df_metrics['predicted'])
        
        # --- Calculate MAE ---
        mae = df_metrics['absolute_error'].mean()

        # --- Calculate WMAPE ---
        sum_absolute_error = df_metrics['absolute_error'].sum()
        sum_actual = df_metrics['actual'].sum()

        wmape_percent = np.nan
        if sum_actual > 0:
            wmape_raw = sum_absolute_error / sum_actual
            wmape_percent = wmape_raw * 100
        else:
            print("Sum of actuals is 0, cannot calculate WMAPE.")

        print("\n--- New Forecast Accuracy Metrics (v5) ---")
        print(f"Mean Absolute Error (MAE): {mae:.4f}")
        print(f"Weighted Mean Absolute Percentage Error (WMAPE): {wmape_percent:.2f}%")
        
        # Save the error file
        error_filename = "forecast_data_with_errors_v5.csv"
        df_metrics.to_csv(error_filename, index=False)
        print(f"Saved detailed data with errors to {error_filename}")
        
    else:
        print("No data with actuals available to calculate metrics.")

except FileNotFoundError:
    print("Error: sales_data.csv not found.")
except Exception as e:
    print(f"An error occurred: {e}")

--- Starting Full Data Generation (v5) with WHOLE NUMBERS ---
Part 1: Creating new expanded 'actuals' data...
Total 'actuals' data rows (original + new): 21850

Part 2: Generating new forecast file 'fake_forecast_data_v5.csv'...
Merged new expanded actuals into forecast timeline.
Applying 'v5' realistic whole number logic to all rows...
Successfully generated 13700 rows.
New data saved to fake_forecast_data_v5.csv

Part 3: Calculating metrics for 'fake_forecast_data_v5.csv'...
Calculating metrics based on 6650 rows where 'actual' is not null.

--- New Forecast Accuracy Metrics (v5) ---
Mean Absolute Error (MAE): 12.4570
Weighted Mean Absolute Percentage Error (WMAPE): 11.50%
Saved detailed data with errors to forecast_data_with_errors_v5.csv


C:\Users\sudee\AppData\Local\Temp\ipykernel_16524\2813935072.py:99: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  final_df['created_at'] = datetime.utcnow()
